**Name:** YOLOv8s Object Detection (Corrected Workflow)
**Student:** Ummadisetti Sumasri 
**Classes:** bottle, can, cup

This notebook provides a clean workflow for training two experiments, evaluating the Roboflow test split and the held-out personal test split, and generating an eight-row error-analysis table from actual predictions.

**Important:** the supplied personal annotations contain only 50 boxes across 43 photos, below the stated minimum of 150. This notebook does not fabricate labels. Add genuine annotations / additional photos and update the dataset ZIP before claiming the requirement is satisfied.

The personal images are split before training: 25 train, 9 validation, 9 test. The personal test images are not copied into training. The model must be retrained from the original Roboflow dataset plus only the personal training split.


## 1. Upload the corrected personal dataset ZIP
Upload `YOLO_Assignment5_Clean_Personal_Splits.zip` when prompted. It contains `personal_dataset/images/{train,val,test}` and matching YOLO labels. The held-out personal test split contains 9 images; small sample size limits conclusions.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose YOLO_Assignment5_Clean_Personal_Splits.zip


In [ ]:
import os, zipfile, glob, shutil, json, random, math
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import torch
from getpass import getpass
from ultralytics import YOLO

SEED = 257058
EPOCHS = 46
IMG_SIZE = 512
BATCH_SIZE = 16
CONF = 0.28
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

zip_path = next((p for p in uploaded if p.lower().endswith(".zip")), None)
if zip_path is None:
    raise FileNotFoundError("Upload the personal split ZIP file.")
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")

PERSONAL = Path("/content/personal_dataset")
for split in ["train","val","test"]:
    assert (PERSONAL/"images"/split).exists(), f"Missing personal images/{split}"
    assert (PERSONAL/"labels"/split).exists(), f"Missing personal labels/{split}"
    print(split, "images:", len(list((PERSONAL/"images"/split).glob("*"))))


## 2. Download the public Roboflow dataset
Enter your Roboflow API key privately when prompted. Do not write it directly into the notebook or publish it to GitHub.

In [ ]:
!pip -q install ultralytics roboflow pyyaml pandas matplotlib opencv-python
from roboflow import Roboflow
api_key = getpass("Roboflow API key (hidden): ")
rf = Roboflow(api_key=api_key)
project = rf.workspace("ai-object-detection").project("bottles-and-cups-detection-6u8tg")
dataset = project.version(27).download("yolov8")
RF = Path(dataset.location)
print("Roboflow dataset:", RF)


## 3. Build training datasets without personal-test leakage
Roboflow's original train/validation/test folders remain intact. Only the personal **train** split is appended to Roboflow training; personal validation and test images are kept separate. The personal test split is never used for model fitting.

In [ ]:
# Create a combined dataset; prefix personal filenames to avoid collisions.
COMBINED = Path("/content/combined_clean")
for split in ["train","val","test"]:
    (COMBINED/"images"/split).mkdir(parents=True, exist_ok=True)
    (COMBINED/"labels"/split).mkdir(parents=True, exist_ok=True)

def copy_split(src_split, dst_split, src_name):
    for folder in ["images","labels"]:
        src = RF/src_name/folder
        dst = COMBINED/folder/dst_split
        for f in src.iterdir():
            if f.is_file():
                shutil.copy2(f, dst/f.name)

copy_split("train","train","train")
copy_split("val","val","valid")
copy_split("test","test","test")

# Add only personal training images/labels to the combined training split.
for folder in ["images","labels"]:
    for f in (PERSONAL/folder/"train").iterdir():
        if f.is_file():
            shutil.copy2(f, COMBINED/folder/"train"/f.name)

cfg = {"path":str(COMBINED),"train":"images/train","val":"images/val","test":"images/test",
       "names":{0:"bottle",1:"can",2:"cup"}}
with open(COMBINED/"data.yaml","w") as f: yaml.safe_dump(cfg,f,sort_keys=False)
print("Combined train:",len(list((COMBINED/"images/train").iterdir())))
print("Roboflow test:",len(list((COMBINED/"images/test").iterdir())))
print("Personal test kept separately:",len(list((PERSONAL/"images/test").iterdir())))


## 4. Experiment 1 — baseline augmentation
The model is retrained from pretrained `yolov8s.pt`. This is a new leakage-free run, so do not copy earlier personal-test metrics into the corrected results.

In [ ]:
model1 = YOLO("yolov8s.pt")
model1.train(data=str(COMBINED/"data.yaml"), epochs=EPOCHS, imgsz=IMG_SIZE,
             batch=BATCH_SIZE, seed=SEED, project="/content/runs/detect",
             name="experiment1_baseline_clean", exist_ok=True)
BEST1 = "/content/runs/detect/experiment1_baseline_clean/weights/best.pt"
assert os.path.exists(BEST1), "Experiment 1 best.pt not found"


## 5. Experiment 2 — Mosaic disabled
This is a controlled comparison: the main settings stay the same while Mosaic augmentation is disabled. Run to completion before making a final numerical comparison.

In [ ]:
model2 = YOLO("yolov8s.pt")
model2.train(data=str(COMBINED/"data.yaml"), epochs=EPOCHS, imgsz=IMG_SIZE,
             batch=BATCH_SIZE, seed=SEED, mosaic=0.0,
             project="/content/runs/detect", name="experiment2_no_mosaic_clean",
             exist_ok=True)
BEST2 = "/content/runs/detect/experiment2_no_mosaic_clean/weights/best.pt"
assert os.path.exists(BEST2), "Experiment 2 best.pt not found"


## 6. Evaluate both experiments on untouched test splits
Roboflow test metrics and personal test metrics are reported separately. Personal test has only 9 images, so interpret it cautiously.

In [ ]:
def evaluate(model_path, data_yaml, split, label):
    model = YOLO(model_path)
    m = model.val(data=data_yaml, split=split, imgsz=IMG_SIZE, batch=BATCH_SIZE,
                  conf=CONF, plots=True, project="/content/runs/detect",
                  name=label, exist_ok=True)
    p, r = float(m.box.mp), float(m.box.mr)
    return {"evaluation":label, "precision":p, "recall":r,
            "f1":2*p*r/(p+r+1e-12), "mAP50":float(m.box.map50),
            "mAP50_95":float(m.box.map)}

# Personal test YAML points only to the held-out personal test split.
personal_yaml = PERSONAL/"data.yaml"
with open(personal_yaml,"w") as f:
    yaml.safe_dump({"path":str(PERSONAL),"train":"images/train","val":"images/val",
                    "test":"images/test","names":{0:"bottle",1:"can",2:"cup"}},f,sort_keys=False)

rows = []
for exp, path in [("Experiment 1",BEST1),("Experiment 2",BEST2)]:
    rows.append(evaluate(path, str(COMBINED/"data.yaml"), "test", exp+"_Roboflow_test"))
    rows.append(evaluate(path, str(personal_yaml), "test", exp+"_Personal_test"))
metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4))
metrics_df.to_csv("/content/experiment_comparison.csv",index=False)


## 7. Build an eight-error table from real predictions
The code below uses Experiment 1 on the held-out personal test set. It matches predictions to ground-truth boxes by class and IoU >= 0.50. It lists up to eight image-level error cases found by the algorithm. If fewer than eight distinct images contain errors, the output will contain fewer rows rather than inventing errors.

In [ ]:
import cv2
from collections import defaultdict

def iou_xyxy(a,b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]); y2=min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    area_a=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    area_b=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/(area_a+area_b-inter+1e-12)

names={0:"bottle",1:"can",2:"cup"}
err_rows=[]
test_imgs=sorted([p for p in (PERSONAL/"images"/"test").iterdir()
                  if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"}])
model=YOLO(BEST1)
for img_path in test_imgs:
    label_path=PERSONAL/"labels"/"test"/(img_path.stem+".txt")
    gt=[]
    if label_path.exists():
        im=cv2.imread(str(img_path)); h,w=im.shape[:2]
        for line in label_path.read_text().splitlines():
            parts=line.split()
            if len(parts)!=5: continue
            c,xc,yc,bw,bh=map(float,parts)
            gt.append({"cls":int(c),"box":[(xc-bw/2)*w,(yc-bh/2)*h,(xc+bw/2)*w,(yc+bh/2)*h],"matched":False})
    pred_result=model.predict(source=str(img_path),imgsz=IMG_SIZE,conf=CONF,verbose=False)[0]
    preds=[]
    for b in pred_result.boxes:
        preds.append({"cls":int(b.cls[0].item()),"conf":float(b.conf[0].item()),
                      "box":[float(v) for v in b.xyxy[0].tolist()],"matched":False})
    # Greedy highest-IoU matching, requiring same class and IoU >= .50
    candidates=[]
    for gi,g in enumerate(gt):
        for pi,p in enumerate(preds):
            if g["cls"]==p["cls"]:
                ov=iou_xyxy(g["box"],p["box"])
                if ov>=0.50: candidates.append((ov,gi,pi))
    for ov,gi,pi in sorted(candidates,reverse=True):
        if not gt[gi]["matched"] and not preds[pi]["matched"]:
            gt[gi]["matched"]=True; preds[pi]["matched"]=True
    missed=[g for g in gt if not g["matched"]]
    extra=[p for p in preds if not p["matched"]]
    if missed or extra:
        if missed and extra: etype="Missed ground-truth object(s) and extra prediction(s)"
        elif missed: etype="False negative / missed object"
        else: etype="False positive / unmatched prediction"
        err_rows.append({"image":img_path.name,"error_type":etype,
                         "missed_classes":", ".join(names[g["cls"]] for g in missed) or "—",
                         "extra_predictions":", ".join(f'{names[p["cls"]]} ({p["conf"]:.2f})' for p in extra) or "—",
                         "note":"Ground-truth matching uses same class and IoU >= 0.50."})
errors=pd.DataFrame(err_rows).head(8)
display(errors)
errors.to_csv("/content/eight_error_analysis.csv",index=False)
print("Distinct images with errors found:",len(err_rows),
      "| rows shown:",len(errors),
      "| do not claim eight if fewer were found.")


## 8. Conclusion/report values
Use the generated `experiment_comparison.csv` and `eight_error_analysis.csv` to fill your final report. Do not use the old personal-test score (P=R=1.0) as a corrected independent result. Current personal labels total 50 boxes; this remains below the 150-box requirement until you add genuine annotations or additional labeled photos. Save the notebook and download the generated CSVs after the runs finish.